In [19]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import os

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
DATASET_PATH = "dataset"

In [20]:
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    color_mode="rgb"
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    color_mode="rgb"
)

Found 1504 files belonging to 4 classes.
Using 1204 files for training.
Found 1504 files belonging to 4 classes.
Using 300 files for validation.


In [21]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [22]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

In [32]:
class_names = sorted([d for d in os.listdir(DATASET_PATH) if os.path.isdir(os.path.join(DATASET_PATH, d))])
num_classes = len(class_names)

base_model = tf.keras.applications.MobileNetV2(
    include_top=False,
    weights='imagenet',
    input_shape=(224, 224, 3)
)

base_model.trainable = False

inputs = layers.Input(shape=(224, 224, 3))

x = layers.Rescaling(1./255)(inputs)
x = data_augmentation(x)

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)

x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)

outputs = layers.Dense(num_classes, activation='softmax')(x)

model = models.Model(inputs, outputs)

In [27]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [28]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

Epoch 1/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 42s 870ms/step - accuracy: 0.4759 - loss: 1.5412 - val_accuracy: 0.8200 - val_loss: 0.6313
Epoch 2/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 30s 781ms/step - accuracy: 0.7118 - loss: 0.8046 - val_accuracy: 0.9267 - val_loss: 0.3696
Epoch 3/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 28s 726ms/step - accuracy: 0.7816 - loss: 0.6037 - val_accuracy: 0.9333 - val_loss: 0.2555
Epoch 4/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 25s 664ms/step - accuracy: 0.8447 - loss: 0.4723 - val_accuracy: 0.9433 - val_loss: 0.1985
Epoch 5/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 25s 662ms/step - accuracy: 0.8497 - loss: 0.4121 - val_accuracy: 0.9533 - val_loss: 0.1697
Epoch 6/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 25s 665ms/step - accuracy: 0.8555 - loss: 0.4178 - val_accuracy: 0.9500 - val_loss: 0.1495
Epoch 7/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 25s 674ms/step - accuracy: 0.8738 - loss: 0.3563 - val_accuracy: 0.9533 - val_loss: 0.1395
Epoch 8/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 26s 691ms/step - accuracy: 0.8796 - loss: 0.3255 - val_accu

In [29]:
import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

plt.figure()
plt.plot(acc, label='train')
plt.plot(val_acc, label='val')
plt.legend()

plt.savefig("plot.png")
print("Saved plot.png")

Saved plot.png


In [30]:
model.save("animal_model.h5")

In [35]:
from tensorflow.keras.preprocessing import image

def predict(img_path):
    img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)

    predictions = model.predict(img_array)[0]

    for i, name in enumerate(class_names):
        print(f"{name}: {predictions[i]:.2f}")

    print(" Final:", class_names[np.argmax(predictions)])

# test
predict("dataset/zebra/002.jpg")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
buffalo: 0.36
elephant: 0.19
rhino: 0.09
zebra: 0.37
 Final: zebra
